# DermArbiter — DermAbench v1-lite Benchmark (Colab A100)

Runs the plan-faithful **hybrid panel** (Specialist=Gemini-2.5-flash · 
Generalist=MedGemma-4B-it 4bit · Skeptic=Qwen3-8B 4bit · Moderator=Gemini) 
over the curated **240-case** DermAbench v1-lite silver set, then scores all 
**8 dimensions**.

**Before you start:**
1. Runtime → Change runtime type → **A100 GPU** (High-RAM).
2. Add Colab Secrets (🔑): `HF_TOKEN`, `GOOGLE_API_KEY`.
3. Upload to `MyDrive/dermarbiter/`:
   - `dermabench_v1lite_bundle.zip`  (make locally: `python scripts/make_colab_bundle.py`)
   - `chroma_cases_derm1m_50k.zip`  (CaseRAG, from notebook 05 — needed for grounding)

Expected wall-clock: **~9–17 h** for 240 cases. The run **checkpoints per case** 
(`--resume`), so you can re-run this notebook to continue after a disconnect.


## 1. Clone repo + install + check A100


In [ ]:
REPO_URL = "https://github.com/Furkanahii/DermArbiter.git"
REPO_BRANCH = "dermabench-v1lite"   # DermAbench commits live here until merged to main

import os
if not os.path.isdir("/content/DermArbiter"):
    !git clone -b $REPO_BRANCH $REPO_URL /content/DermArbiter
else:
    !cd /content/DermArbiter && git fetch origin && git checkout $REPO_BRANCH && git pull
%cd /content/DermArbiter

!pip install -q -e .
!pip install -q huggingface_hub bitsandbytes accelerate


In [ ]:
from google.colab import userdata
import os
for key in ("HF_TOKEN", "GOOGLE_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
        print(f"✅ {key} loaded ({len(os.environ[key])} chars)")
    except userdata.SecretNotFoundError:
        raise RuntimeError(f"{key} secret missing — add it via the 🔑 panel.")
os.environ.setdefault("HUGGINGFACE_HUB_TOKEN", os.environ["HF_TOKEN"])

import torch
assert torch.cuda.is_available(), "No CUDA — set Runtime → A100 GPU."
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch CUDA {torch.version.cuda} · {torch.cuda.get_device_name(0)}")


## 2. Mount Drive + restore the v1-lite bundle (gold + 240 images)

The curated gold + images are gitignored, so we ship the exact validated 
set via Drive instead of re-curating on Colab.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/dermarbiter")
bundle = DRIVE / "dermabench_v1lite_bundle.zip"
assert bundle.exists(), f"{bundle} not found — upload it (make with scripts/make_colab_bundle.py)."

# Unzips to repo-relative paths: data/dermabench/dermabench_v1lite.jsonl + raw/.../images/*.png
!unzip -q -o {str(bundle)} -d /content/DermArbiter

import json, os
GOLD = "data/dermabench/dermabench_v1lite.jsonl"
cases = [json.loads(l) for l in open(GOLD)]
present = sum(1 for c in cases if c.get('image_path') and os.path.exists(c['image_path']))
print(f"✅ v1-lite restored: {len(cases)} cases, {present} images present")
assert present == len(cases), "Some images missing — rebuild the bundle locally."


## 3. Restore CaseRAG (grounding dimension)

Dimension 8 (grounding) needs the retrieval index so agents can cite evidence 
cards. Without it the run still completes but grounding scores ~0.


In [ ]:
case_zip = DRIVE / "chroma_cases_derm1m_50k.zip"
if case_zip.exists():
    if not Path("data/chroma_cases").exists():
        !mkdir -p data && cd data && unzip -q {str(case_zip)}
    print("✅ CaseRAG restored at data/chroma_cases")
else:
    print("⚠️  CaseRAG zip not in Drive — grounding (Dim 8) will score ~0. Run notebook 05 to build it.")


## 4. (optional) PanDerm weights — visual classifier tool
Set the two URLs (or upload to `weights/`). Without them the PanDerm tool 
is skipped; the other 8 tools still run.


In [ ]:
import os; from pathlib import Path
Path("weights").mkdir(exist_ok=True)
W, H = os.environ.get("PANDERM_WEIGHT_URL",""), os.environ.get("PANDERM_HEAD_URL","")
if (Path("weights/panderm.pth").exists() and Path("weights/panderm_head.pth").exists()):
    print("✅ PanDerm weights present.")
elif W and H:
    !wget -q -O weights/panderm.pth "$W" && wget -q -O weights/panderm_head.pth "$H"
    print("✅ PanDerm weights downloaded.")
else:
    print("⚠️  PanDerm weights not set — that tool will be skipped.")


## 5. Smoke test — 2 cases first (catch config/model errors early)
Always run this before the full sweep. If it produces predictions, the 
hybrid panel + DermAbench bridge are wired correctly.


In [ ]:
!python scripts/run_dermagent_subset.py \
    --real \
    --subset data/dermabench/dermabench_v1lite.jsonl \
    --dermabench-out results/dermabench/predictions_smoke.jsonl \
    --output-dir results/dermabench/smoke \
    --max-cases 2 -v


## 6. Full run — all 240 cases (checkpointed, resumable)

Writes predictions per case with fsync, so a disconnect loses nothing: just 
re-run this cell and it continues via `--resume`. **~9–17 h.**


In [ ]:
PRED = "results/dermabench/predictions_v1lite.jsonl"
METRICS_DIR = "results/dermabench/v1lite"
import os
# --resume points at the metrics JSONL; if present, completed cases are skipped.
resume_flag = ""
import glob
done = glob.glob(METRICS_DIR + "/*.jsonl")
if done:
    resume_flag = "--resume " + done[0]
    print("Resuming from", done[0])

!python scripts/run_dermagent_subset.py \
    --real \
    --subset data/dermabench/dermabench_v1lite.jsonl \
    --dermabench-out {PRED} \
    --output-dir {METRICS_DIR} \
    {resume_flag}


## 7. Score all 8 DermAbench dimensions


In [ ]:
from dermarbiter.evaluation.dermabench import DermAbenchScorer
GOLD = "data/dermabench/dermabench_v1lite.jsonl"
PRED = "results/dermabench/predictions_v1lite.jsonl"
scorer = DermAbenchScorer.from_jsonl(GOLD, PRED)
scorer.print_report()

import json
report = scorer.score_all()
with open("results/dermabench/v1lite_scores.json", "w") as f:
    json.dump(report, f, indent=2)
print("\nComposite:", round(report["composite"], 3), "over", report["n_cases"], "cases")


## 8. Persist results to Drive


In [ ]:
import shutil; from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/dermarbiter"); DRIVE.mkdir(parents=True, exist_ok=True)
for src in ["results/dermabench/predictions_v1lite.jsonl",
            "results/dermabench/v1lite_scores.json"]:
    p = Path(src)
    if p.exists():
        shutil.copy(p, DRIVE / p.name)
        print("✅", src, "→ Drive")
# also back up the per-case metrics JSONL (the resume checkpoint)
import glob
for m in glob.glob("results/dermabench/v1lite/*.jsonl"):
    shutil.copy(m, DRIVE / Path(m).name)
    print("✅", m, "→ Drive")


## Next steps
- Inspect `v1lite_scores.json` per-dimension; check the **fairness gap** and 
  **safety triage_sensitivity** (60 malignant, 16 dark-skin).
- Compare against an ablation (same base model, no scaffolding) and a GPT-4o 
  baseline once the key lands.
- Scale to v1 (~1000 cases) per `DERMABENCH_SCALING_PLAN.md`.
